# vLLM Hybrid Benchmark Analysis

GPU-only vs Hybrid (GPU+CPU) performance comparison dashboard  
Time-series monitoring, per-run comparison, and cross-model analysis

In [ ]:
# ============================================================
#  Configuration — edit here only
# ============================================================
RESULTS_DIR = "/mnt/d/projects/vllm_hybrid/eval/results"
# ============================================================

import json, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

GPU_COLOR    = "#4C72B0"
HYBRID_COLOR = "#DD8452"
PALETTE      = {"GPU-only": GPU_COLOR, "Hybrid": HYBRID_COLOR}

RESULTS_DIR = Path(RESULTS_DIR)
print(f"Results directory: {RESULTS_DIR}")


In [ ]:
# ============================================================
#  Load experiment runs
# ============================================================

def parse_ts(name):
    try:    return datetime.strptime(name, "%Y%m%d_%H%M%S")
    except: return None

def load_run(run_dir: Path):
    run = {"dir": run_dir, "name": run_dir.name, "ts": parse_ts(run_dir.name)}
    for mode in ("gpu_only", "hybrid"):
        jf = run_dir / f"{mode}.json"
        if jf.exists():
            with open(jf) as f: run[mode] = json.load(f)
        for suffix in ("monitor_gpu.csv", "monitor_cpu.csv"):
            cf = run_dir / f"{mode}_{suffix}"
            if cf.exists():
                df = pd.read_csv(cf)
                df["timestamp"] = pd.to_datetime(df["timestamp"])
                run[f"{mode}_{suffix.replace('.csv','')}"] = df
    cj = run_dir / "comparison.json"
    if cj.exists():
        with open(cj) as f: run["comparison"] = json.load(f)
    return run

runs = []
for d in sorted(RESULTS_DIR.iterdir()):
    if d.is_symlink() or not d.is_dir(): continue
    ts = parse_ts(d.name)
    if ts is None: continue
    runs.append(load_run(d))

cmp_runs = [r for r in runs if "comparison" in r]

print(f"Total runs    : {len(runs)}")
print(f"With comparison: {len(cmp_runs)}")
for r in runs:
    has   = [k for k in ("gpu_only","hybrid","comparison") if k in r]
    model = r.get("hybrid", r.get("gpu_only",{})).get("model_id","?").split("/")[-1]
    prompts = r.get("hybrid", r.get("gpu_only",{})).get("num_prompts","?")
    print(f"  {r['name']}  model={model:30s}  prompts={str(prompts):5}  data={has}")


In [ ]:
# ============================================================
#  Section 1 ▶ All Runs Summary Table
# ============================================================

rows = []
for r in runs:
    for mode in ("gpu_only", "hybrid"):
        d = r.get(mode)
        if d is None: continue
        rows.append({
            "Run": r["name"], "Mode": mode,
            "Model": d.get("model_id","?").split("/")[-1],
            "Prompts": d.get("num_prompts"),
            "Completed": d.get("completed"),
            "Duration (s)": round(d.get("duration",0),1),
            "Req TP (req/s)": round(d.get("request_throughput",0),2),
            "Out TP (tok/s)": round(d.get("output_throughput",0),1),
            "Total TP": round(d.get("total_token_throughput",0),1),
            "Mean TTFT (ms)": round(d.get("mean_ttft_ms",0),1),
            "P99 TTFT (ms)": round(d.get("p99_ttft_ms",0),1),
            "Mean TPOT (ms)": round(d.get("mean_tpot_ms",0),2),
            "P99 TPOT (ms)": round(d.get("p99_tpot_ms",0),2),
        })

summary_df = pd.DataFrame(rows).sort_values(["Run","Mode"])

def style_summary(df):
    def row_bg(row):
        c = "#e8f5e9" if row["Mode"]=="hybrid" else "#fffde7"
        return [f"background-color:{c}"]*len(row)
    return (df.style
        .apply(row_bg, axis=1)
        .format({"Req TP (req/s)":"{:.2f}","Out TP (tok/s)":"{:.0f}","Total TP":"{:.0f}",
                 "Mean TTFT (ms)":"{:.0f}","P99 TTFT (ms)":"{:.0f}",
                 "Mean TPOT (ms)":"{:.2f}","P99 TPOT (ms)":"{:.2f}"})
        .set_table_styles([
            {"selector":"th","props":[("background-color","#2d4a7a"),("color","white"),
                                      ("font-size","12px"),("padding","5px 10px")]},
            {"selector":"tr:nth-child(even)","props":[("background-color","#fafafa")]},
            {"selector":"tr:hover","props":[("background-color","#eef3fb")]},
        ])
        .set_caption("<b style=\"font-size:14px\">All Runs Summary — yellow=GPU-only, green=Hybrid</b>"))

style_summary(summary_df)


In [ ]:
# ============================================================
#  Section 2 ▶ Time-Series: Throughput Trend
# ============================================================

ts_list, req_gpu, req_hyb, out_gpu, out_hyb, models = [], [], [], [], [], []
for r in cmp_runs:
    ts_list.append(r["ts"])
    g = r.get("gpu_only", {}); h = r.get("hybrid", {})
    req_gpu.append(g.get("request_throughput", np.nan))
    req_hyb.append(h.get("request_throughput", np.nan))
    out_gpu.append(g.get("output_throughput",  np.nan))
    out_hyb.append(h.get("output_throughput",  np.nan))
    models.append(h.get("model_id", g.get("model_id","?")).split("/")[-1])

xlabels = [f"{t.strftime('%m/%d %H:%M')}\n{m}" for t, m in zip(ts_list, models)]
x = range(len(ts_list))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Throughput Trend over Runs", fontsize=13, fontweight="bold")

for ax, gpu_v, hyb_v, title, ylabel in zip(
    axes,
    [req_gpu, out_gpu], [req_hyb, out_hyb],
    ["Request Throughput", "Output Token Throughput"],
    ["req/s", "tok/s"]
):
    ax.plot(x, gpu_v, "o-", color=GPU_COLOR,    label="GPU-only", linewidth=2, markersize=8, zorder=3)
    ax.plot(x, hyb_v, "s--", color=HYBRID_COLOR, label="Hybrid",   linewidth=2, markersize=8, zorder=3)
    ax.fill_between(x, gpu_v, alpha=0.08, color=GPU_COLOR)
    ax.fill_between(x, hyb_v, alpha=0.08, color=HYBRID_COLOR)
    ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=8)
    ax.set_ylabel(ylabel); ax.set_title(title, fontweight="semibold")
    ax.legend(framealpha=0.8)

sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
#  Section 3 ▶ Time-Series: Latency Trend
# ============================================================

lat_series = [
    ("mean_ttft_ms", "Mean TTFT"),
    ("mean_tpot_ms", "Mean TPOT"),
    ("mean_itl_ms",  "Mean ITL"),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Latency Trend over Runs", fontsize=13, fontweight="bold")

for ax, (key, title) in zip(axes, lat_series):
    gpu_v = [r.get("gpu_only",{}).get(key, np.nan) for r in cmp_runs]
    hyb_v = [r.get("hybrid",  {}).get(key, np.nan) for r in cmp_runs]
    ax.plot(x, gpu_v, "o-",  color=GPU_COLOR,    label="GPU-only", linewidth=2, markersize=8, zorder=3)
    ax.plot(x, hyb_v, "s--", color=HYBRID_COLOR, label="Hybrid",   linewidth=2, markersize=8, zorder=3)
    ax.fill_between(x, gpu_v, alpha=0.08, color=GPU_COLOR)
    ax.fill_between(x, hyb_v, alpha=0.08, color=HYBRID_COLOR)
    ax.set_xticks(x); ax.set_xticklabels(xlabels, fontsize=7)
    ax.set_ylabel("ms"); ax.set_title(title, fontweight="semibold")
    ax.legend(framealpha=0.8)

sns.despine()
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
#  Section 4 ▶ Hybrid vs GPU-only: Speedup & Improvement
# ============================================================

metrics_plot = [
    ("request_throughput", "Request TP",  True),
    ("output_throughput",  "Output TP",   True),
    ("mean_ttft_ms",       "Mean TTFT",   False),
    ("mean_tpot_ms",       "Mean TPOT",   False),
    ("mean_itl_ms",        "Mean ITL",    False),
    ("p99_ttft_ms",        "P99 TTFT",    False),
]

diff_records = []
for r in cmp_runs:
    csub  = r.get("comparison",{}).get("comparison",{})
    g = r.get("gpu_only",{}); h = r.get("hybrid",{})
    model = h.get("model_id", g.get("model_id","?")).split("/")[-1]
    row = {"Run": r["name"], "Model": model,
           "Speedup": csub.get("output_tok_speedup", np.nan)}
    for key, label, _ in metrics_plot:
        entry = csub.get(key,{})
        row[label+"_diff"] = entry.get("diff_pct", np.nan) if isinstance(entry,dict) else np.nan
    diff_records.append(row)

diff_df = pd.DataFrame(diff_records)
run_xlabels = [f"{r['name'][4:]}\n{r.get('hybrid',r.get('gpu_only',{})).get('model_id','?').split('/')[-1]}" for r in cmp_runs]

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Hybrid vs GPU-only: Speedup & Per-Metric Improvement",
             fontsize=13, fontweight="bold")

# Speedup bar
ax = axes[0]
speedups = diff_df["Speedup"].tolist()
colors   = [sns.color_palette("muted")[2] if s>=1.0 else sns.color_palette("muted")[3] for s in speedups]
bars = ax.bar(range(len(speedups)), speedups, color=colors, edgecolor="white", linewidth=0.8, width=0.55)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1.5, label="Baseline (GPU-only = 1.0x)")
ax.set_xticks(range(len(run_xlabels))); ax.set_xticklabels(run_xlabels, fontsize=8)
ax.set_ylabel("Output Token Speedup (×)"); ax.set_title("Output Token Throughput Speedup", fontweight="semibold")
ax.legend(framealpha=0.8)
for bar, v in zip(bars, speedups):
    ax.text(bar.get_x()+bar.get_width()/2, v+0.008, f"{v:.3f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

# Improvement bar (sign-adjusted: positive = Hybrid wins)
ax = axes[1]
metric_keys  = [l+"_diff" for _, l, _ in metrics_plot]
metric_labels= [l for _, l, _ in metrics_plot]
higher_flags = [h for _, _, h in metrics_plot]

n_runs = len(diff_df)
bar_w  = 0.7 / n_runs
offsets= np.linspace(-0.35+bar_w/2, 0.35-bar_w/2, n_runs)
pal    = sns.color_palette("tab10", n_runs)

for i, (_, row) in enumerate(diff_df.iterrows()):
    adjusted = [-row[k] if not hb else row[k]
                for k, hb in zip(metric_keys, higher_flags)]
    ax.bar([j+offsets[i] for j in range(len(metric_keys))],
           adjusted, width=bar_w, label=row["Run"][4:], color=pal[i], alpha=0.85)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(range(len(metric_labels))); ax.set_xticklabels(metric_labels, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Improvement (%) — positive = Hybrid wins")
ax.set_title("Per-Metric Hybrid Improvement", fontweight="semibold")
ax.legend(fontsize=7, ncol=2, framealpha=0.8)

sns.despine()
plt.tight_layout()
plt.show()

# Summary table
display_diff = diff_df[["Run","Model","Speedup"]+metric_keys].copy()
display_diff.columns = ["Run","Model","Speedup"]+metric_labels
display_diff.style \
    .format({c:"{:+.1f}%" for c in metric_labels}|{"Speedup":"{:.3f}x"}) \
    .background_gradient(subset=["Speedup"], cmap="RdYlGn", vmin=0.5, vmax=1.5) \
    .set_table_styles([{"selector":"th","props":[("background-color","#2d4a7a"),
                                                  ("color","white"),("font-size","12px"),
                                                  ("padding","5px 10px")]}]) \
    .set_caption("<b style=\"font-size:14px\">Hybrid Improvement Summary</b>")


In [ ]:
# ============================================================
#  Section 5 ▶ GPU Time-Series Monitoring (all runs)
# ============================================================

gpu_series = [
    ("gpu_avg_util_pct",     "GPU Utilization (%)",  (0, 105)),
    ("gpu_avg_mem_util_pct", "GPU Memory Util (%)",  (0, 105)),
    ("gpu_avg_power_w",      "GPU Power (W)",        (None, None)),
]

def plot_gpu_ts(run):
    model = run.get("hybrid", run.get("gpu_only",{})).get("model_id","?").split("/")[-1]
    fig, axes = plt.subplots(3, 2, figsize=(16, 11), sharex="col")
    fig.suptitle(f"{run['name']}  [{model}]  GPU Time-Series",
                 fontsize=13, fontweight="bold")

    for col, (df, color, label) in enumerate([
        (run.get("gpu_only_monitor_gpu"), GPU_COLOR,    "GPU-only"),
        (run.get("hybrid_monitor_gpu"),   HYBRID_COLOR, "Hybrid"),
    ]):
        for row, (cname, ylabel, ylim) in enumerate(gpu_series):
            ax = axes[row][col]
            if df is None or cname not in df.columns:
                ax.text(0.5,0.5,"N/A",ha="center",va="center",
                        transform=ax.transAxes,fontsize=12,color="#aaa"); continue
            t  = df["elapsed_s"]
            vs = df[cname]
            ax.plot(t, vs, color=color, linewidth=1.3, zorder=3)
            ax.fill_between(t, vs, alpha=0.15, color=color, zorder=2)
            ax.plot(t, vs.rolling(5,center=True).mean(),
                    color=color, linewidth=2.2, linestyle="--", alpha=0.55, zorder=4)
            ax.axhline(vs.dropna().mean(), color=color, linewidth=0.9, linestyle=":", alpha=0.7)
            if ylim[0] is not None: ax.set_ylim(*ylim)
            ax.set_ylabel(ylabel, fontsize=9)
            ax.set_title(f"{label} — {ylabel}", fontsize=10, fontweight="semibold")
            if row==2: ax.set_xlabel("Elapsed (s)", fontsize=9)
            v = vs.dropna()
            ax.text(0.02,0.96,
                    f"mean={v.mean():.1f}  max={v.max():.1f}  min={v.min():.1f}  std={v.std():.1f}",
                    transform=ax.transAxes, va="top", fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.3",facecolor="white",edgecolor="#ddd",alpha=0.85))

    sns.despine(); plt.tight_layout(); plt.show()

for r in runs:
    if r.get("gpu_only_monitor_gpu") is not None or r.get("hybrid_monitor_gpu") is not None:
        plot_gpu_ts(r)


In [ ]:
# ============================================================
#  Section 6 ▶ CPU Time-Series Monitoring (all runs)
# ============================================================

CORE_COLS = [f"core{i}_util_pct" for i in range(16)]

def plot_cpu_ts(run):
    model = run.get("hybrid", run.get("gpu_only",{})).get("model_id","?").split("/")[-1]
    fig, axes = plt.subplots(3, 2, figsize=(16, 13))
    fig.suptitle(f"{run['name']}  [{model}]  CPU Time-Series",
                 fontsize=13, fontweight="bold")

    for col, (df, color, label, cmap_name) in enumerate([
        (run.get("gpu_only_monitor_cpu"), GPU_COLOR,    "GPU-only", "YlGnBu"),
        (run.get("hybrid_monitor_cpu"),   HYBRID_COLOR, "Hybrid",   "YlOrRd"),
    ]):
        if df is None:
            for ax in axes[:,col]:
                ax.text(0.5,0.5,"N/A",ha="center",va="center",
                        transform=ax.transAxes,fontsize=12,color="#aaa")
            continue
        t  = df["elapsed_s"]
        cc = [c for c in CORE_COLS if c in df.columns]

        # Row 0: avg utilization
        ax = axes[0][col]
        v  = df["cpu_avg_util_pct"]
        ax.plot(t, v, color=color, linewidth=1.5, zorder=3)
        ax.fill_between(t, v, alpha=0.15, color=color, zorder=2)
        ax.plot(t, v.rolling(5,center=True).mean(),
                color=color, linewidth=2.2, linestyle="--", alpha=0.55, zorder=4)
        ax.axhline(v.mean(), color=color, linewidth=0.9, linestyle=":", alpha=0.7)
        ax.set_ylim(0,105); ax.set_ylabel("CPU Avg Util (%)", fontsize=9)
        ax.set_title(f"{label} — CPU Average Utilization", fontsize=10, fontweight="semibold")
        vs = v.dropna()
        ax.text(0.02,0.96,
                f"mean={vs.mean():.1f}  max={vs.max():.1f}  min={vs.min():.1f}  std={vs.std():.1f}",
                transform=ax.transAxes, va="top", fontsize=8,
                bbox=dict(boxstyle="round,pad=0.3",facecolor="white",edgecolor="#ddd",alpha=0.85))

        # Row 1: per-core heatmap
        ax = axes[1][col]
        if cc:
            im = ax.imshow(df[cc].T.values, aspect="auto", cmap=cmap_name,
                           vmin=0, vmax=100, interpolation="bilinear")
            ax.set_yticks(range(len(cc)))
            ax.set_yticklabels([c.replace("_util_pct","").replace("core","Core ") for c in cc], fontsize=7)
            ax.set_xlabel("Sample Index", fontsize=9)
            ax.set_title(f"{label} — Per-Core Utilization Heatmap", fontsize=10, fontweight="semibold")
            cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
            cb.set_label("Util (%)", fontsize=8)
            for i, mv in enumerate(df[cc].mean()):
                ax.text(df[cc].shape[0]*1.01, i, f"{mv:.0f}%",
                        va="center", fontsize=7, color="#333", fontweight="bold")

        # Row 2: RAM
        ax = axes[2][col]
        if "cpu_mem_used_mb" in df.columns:
            used_gb = df["cpu_mem_used_mb"]/1024
            ax.plot(t, used_gb, color=color, linewidth=1.4, label="Used", zorder=3)
            ax.fill_between(t, used_gb, alpha=0.15, color=color, zorder=2)
        if "cpu_mem_avail_mb" in df.columns:
            ax.plot(t, df["cpu_mem_avail_mb"]/1024,
                    color=color, linewidth=1.0, linestyle="--", alpha=0.5, label="Available")
        ax.set_ylabel("RAM (GB)", fontsize=9); ax.set_xlabel("Elapsed (s)", fontsize=9)
        ax.set_title(f"{label} — RAM Usage", fontsize=10, fontweight="semibold")
        ax.legend(fontsize=8, framealpha=0.7)
        if "cpu_mem_used_mb" in df.columns:
            m = (df["cpu_mem_used_mb"]/1024).dropna()
            ax.text(0.02,0.96,
                    f"mean={m.mean():.1f}GB  max={m.max():.1f}GB  min={m.min():.1f}GB  std={m.std():.1f}GB",
                    transform=ax.transAxes, va="top", fontsize=8,
                    bbox=dict(boxstyle="round,pad=0.3",facecolor="white",edgecolor="#ddd",alpha=0.85))

    sns.despine(); plt.tight_layout(); plt.show()

for r in runs:
    if r.get("gpu_only_monitor_cpu") is not None or r.get("hybrid_monitor_cpu") is not None:
        plot_cpu_ts(r)


In [ ]:
# ============================================================
#  Section 7 ▶ Resource Utilization Comparison (per run)
# ============================================================

for r in cmp_runs:
    c  = r.get("comparison", {})
    gu = c.get("gpu_utilization", {})
    cu = c.get("cpu_utilization", {})
    if not gu or not cu: continue

    model = r.get("hybrid", r.get("gpu_only",{})).get("model_id","?").split("/")[-1]

    categories = ["GPU Util (%)", "GPU Mem Util (%)", "GPU Power (W)"]
    gpu_keys   = ["gpu_avg_util_pct", "gpu_avg_mem_util_pct", "gpu_avg_power_w"]
    mode_labels= ["GPU-only", "Hybrid"]

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    fig.suptitle(f"Resource Utilization: {r['name']}  [{model}]",
                 fontsize=13, fontweight="bold")

    # Top row: GPU metrics
    for j, (cat, gkey) in enumerate(zip(categories, gpu_keys)):
        ax = axes[0][j]
        vals, errs_lo, errs_hi = [], [], []
        for mode in ("gpu_only","hybrid"):
            stat = gu.get(mode,{}).get(gkey,{})
            mv   = stat.get("mean",0)
            vals.append(mv)
            errs_lo.append(mv - stat.get("min", mv))
            errs_hi.append(stat.get("max", mv) - mv)
        sns.barplot(x=mode_labels, y=vals, palette=PALETTE,
                    order=["GPU-only","Hybrid"], ax=ax,
                    edgecolor="white", linewidth=1.2, width=0.5)
        ax.errorbar(mode_labels, vals, yerr=[errs_lo,errs_hi],
                    fmt="none", color="#333", capsize=6, linewidth=1.5)
        ax.set_title(cat, fontweight="semibold"); ax.set_xlabel("")
        for patch, v in zip(ax.patches, vals):
            ax.text(patch.get_x()+patch.get_width()/2,
                    v + max(errs_hi)*0.05, f"{v:.1f}",
                    ha="center", va="bottom", fontsize=10, fontweight="bold")

    # Bottom row: CPU avg util
    for j, (mode, label) in enumerate([("gpu_only","GPU-only"),("hybrid","Hybrid")]):
        ax = axes[1][j]
        stat  = cu.get(mode,{}).get("cpu_avg_util_pct",{})
        mv    = stat.get("mean",0)
        err_lo= mv - stat.get("min",mv)
        err_hi= stat.get("max",mv) - mv
        color = GPU_COLOR if j==0 else HYBRID_COLOR
        ax.bar([label], [mv], color=[color], alpha=0.85, edgecolor="white", linewidth=1.2)
        ax.errorbar([label], [mv], yerr=[[err_lo],[err_hi]],
                    fmt="none", color="#333", capsize=6, linewidth=1.5)
        ax.set_ylim(0,105); ax.set_ylabel("CPU Avg Util (%)"); ax.set_xlabel("")
        ax.set_title(f"{label} — CPU Average Utilization", fontweight="semibold")
        ax.text(0, mv+err_hi+1.5, f"{mv:.1f}%",
                ha="center", va="bottom", fontsize=13, fontweight="bold")

    axes[1][2].axis("off")
    sns.despine(); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 8 ▶ Cross-Model Comparison
# ============================================================

model_groups = {}
for r in cmp_runs:
    m = r.get("hybrid", r.get("gpu_only",{})).get("model_id","?").split("/")[-1]
    model_groups.setdefault(m, []).append(r)

print(f"Model groups: {list(model_groups.keys())}")

if len(model_groups) > 1:
    model_names = list(model_groups.keys())
    sp_req, sp_out, ttft_d, tpot_d = [], [], [], []

    for mn in model_names:
        csub = model_groups[mn][-1].get("comparison",{}).get("comparison",{})
        sp_req.append(csub.get("request_throughput_speedup", np.nan))
        sp_out.append(csub.get("output_tok_speedup", np.nan))
        ttft_d.append(csub.get("mean_ttft_ms",{}).get("diff_pct", np.nan))
        tpot_d.append(csub.get("mean_tpot_ms",{}).get("diff_pct", np.nan))

    cmp_model_df = pd.DataFrame({
        "Model": model_names*2,
        "Metric": ["Request TP Speedup"]*len(model_names)+["Output TP Speedup"]*len(model_names),
        "Value": sp_req + sp_out,
    })
    cmp_lat_df = pd.DataFrame({
        "Model": model_names*2,
        "Metric": ["TTFT Improvement (%)"]*len(model_names)+["TPOT Improvement (%)"]*len(model_names),
        "Value": [-v for v in ttft_d] + [-v for v in tpot_d],
    })

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    fig.suptitle("Cross-Model Comparison (latest run per model)",
                 fontsize=13, fontweight="bold")

    sns.barplot(data=cmp_model_df, x="Model", y="Value", hue="Metric",
                palette=["#4C72B0","#55A868"], ax=axes[0],
                edgecolor="white", linewidth=0.8, width=0.6)
    axes[0].axhline(1.0, color="red", linestyle="--", linewidth=1.5, label="Baseline 1.0x")
    axes[0].set_title("Throughput Speedup", fontweight="semibold")
    axes[0].set_ylabel("Speedup (×)"); axes[0].set_xlabel("")
    axes[0].legend(fontsize=8)

    sns.barplot(data=cmp_lat_df, x="Model", y="Value", hue="Metric",
                palette=["#DD8452","#8172B2"], ax=axes[1],
                edgecolor="white", linewidth=0.8, width=0.6)
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set_title("Latency Improvement — positive = Hybrid wins",
                      fontweight="semibold")
    axes[1].set_ylabel("Improvement (%)"); axes[1].set_xlabel("")
    axes[1].legend(fontsize=8)

    sns.despine(); plt.tight_layout(); plt.show()
else:
    print("Only one model group — cross-model comparison skipped.")


In [ ]:
# ============================================================
#  Section 9 ▶ Latency Distribution: P50 / P99
# ============================================================

lat_groups = [
    ("TTFT", "median_ttft_ms", "mean_ttft_ms", "p99_ttft_ms", "std_ttft_ms"),
    ("TPOT", "median_tpot_ms", "mean_tpot_ms", "p99_tpot_ms", "std_tpot_ms"),
    ("ITL",  "median_itl_ms",  "mean_itl_ms",  "p99_itl_ms",  "std_itl_ms"),
]

lat_rows = []
for name, p50k, meank, p99k, stdk in lat_groups:
    for r in cmp_runs:
        run_label = f"{r['name'][4:]}\n{r.get('hybrid',r.get('gpu_only',{})).get('model_id','?').split('/')[-1]}"
        for mode, src in [("GPU-only", r.get("gpu_only")), ("Hybrid", r.get("hybrid"))]:
            if src is None: continue
            for stat, key in [("P50",p50k),("Mean",meank),("P99",p99k),("Std",stdk)]:
                lat_rows.append({"Metric":name,"Run":run_label,"Mode":mode,
                                  "Stat":stat,"ms":src.get(key,0) or 0})

lat_df = pd.DataFrame(lat_rows)
stat_pal = {"P50":"#4C72B0","Mean":"#64B5CD","P99":"#DD8452","Std":"#CCBBAA"}

g = sns.FacetGrid(lat_df, col="Metric", height=5, aspect=0.95,
                  sharey=False, gridspec_kws={"wspace":0.35})

def _draw_lat(data, **kw):
    ax = plt.gca()
    sub = data[data["Stat"].isin(["P50","P99"])]
    sns.barplot(data=sub, x="Mode", y="ms", hue="Stat",
                palette=stat_pal, order=["GPU-only","Hybrid"],
                hue_order=["P50","P99"],
                edgecolor="white", linewidth=0.8, ax=ax, width=0.6)
    for patch in ax.patches:
        v = patch.get_height()
        if v > 0:
            ax.text(patch.get_x()+patch.get_width()/2, v*1.01, f"{v:.1f}",
                    ha="center", va="bottom", fontsize=7)

g.map_dataframe(_draw_lat)
g.set_axis_labels("", "Latency (ms)")
g.set_titles(col_template="{col_name} Latency — P50 & P99")
g.add_legend(title="Stat", bbox_to_anchor=(1.02,0.5), loc="center left")
g.figure.suptitle("Latency Distribution: P50 / P99 per Run",
                   fontsize=14, fontweight="bold", y=1.03)
sns.despine()
plt.show()


In [ ]:
# ============================================================
#  Section 10 ▶ Power Efficiency (tok/s per Watt)
# ============================================================

eff_rows = []
for r in cmp_runs:
    gu    = r.get("comparison",{}).get("gpu_utilization",{})
    model = r.get("hybrid",r.get("gpu_only",{})).get("model_id","?").split("/")[-1]
    for mode, label in [("gpu_only","GPU-only"),("hybrid","Hybrid")]:
        d    = r.get(mode, {})
        pw   = gu.get(mode,{}).get("gpu_avg_power_w",{}).get("mean", np.nan)
        out  = d.get("output_throughput", np.nan)
        eff_rows.append({"Run":r["name"],"Model":model,"Mode":label,
                         "Output TP (tok/s)":out,"Avg Power (W)":pw,
                         "Efficiency (tok/s/W)":out/pw if pw and pw>0 else np.nan})

eff_df = pd.DataFrame(eff_rows)

eff_long = eff_df.melt(id_vars=["Run","Model","Mode"],
                        value_vars=["Output TP (tok/s)","Avg Power (W)","Efficiency (tok/s/W)"],
                        var_name="Metric", value_name="Value")

g = sns.FacetGrid(eff_long, col="Metric", height=4.5, aspect=0.85,
                  sharey=False, gridspec_kws={"wspace":0.4})

def _draw_eff(data, **kw):
    ax = plt.gca()
    sns.barplot(data=data, x="Mode", y="Value", palette=PALETTE,
                order=["GPU-only","Hybrid"], ax=ax,
                edgecolor="white", linewidth=1.0, width=0.5)
    for patch in ax.patches:
        v = patch.get_height()
        if not np.isnan(v) and v>0:
            ax.text(patch.get_x()+patch.get_width()/2, v*1.015, f"{v:.2f}",
                    ha="center", va="bottom", fontsize=9, fontweight="bold")

g.map_dataframe(_draw_eff)
g.set_axis_labels("", "Value")
g.set_titles(col_template="{col_name}")
g.figure.suptitle("GPU Power Efficiency Analysis",
                   fontsize=14, fontweight="bold", y=1.04)
sns.despine()
plt.show()

# Table
display(eff_df.style
    .format({"Output TP (tok/s)":"{:.1f}","Avg Power (W)":"{:.1f}","Efficiency (tok/s/W)":"{:.3f}"}, na_rep="-")
    .background_gradient(cmap="Greens", subset=["Efficiency (tok/s/W)"])
    .set_table_styles([{"selector":"th","props":[("background-color","#2d4a7a"),("color","white"),
                                                  ("font-size","12px"),("padding","5px 10px")]}])
    .set_caption("<b style=\"font-size:14px\">Power Efficiency Table</b>"))


In [ ]:
# ============================================================
#  Section 11 ▶ Radar Chart — Latest Run
# ============================================================

latest = cmp_runs[-1]
g = latest.get("gpu_only", {}); h = latest.get("hybrid", {})
model_name = h.get("model_id", g.get("model_id","?")).split("/")[-1]

radar_metrics = ["Request TP","Output TP","Total TP","TTFT (inv)","TPOT (inv)","ITL (inv)"]
g_vals = [g.get("request_throughput",0), g.get("output_throughput",0),
          g.get("total_token_throughput",0),
          1/g.get("mean_ttft_ms",1), 1/g.get("mean_tpot_ms",1), 1/g.get("mean_itl_ms",1)]
h_vals = [h.get("request_throughput",0), h.get("output_throughput",0),
          h.get("total_token_throughput",0),
          1/h.get("mean_ttft_ms",1), 1/h.get("mean_tpot_ms",1), 1/h.get("mean_itl_ms",1)]

safe = lambda a, b: a/b if b else 1.0
g_norm = [1.0]*len(g_vals)
h_norm = [safe(h_vals[i],g_vals[i]) if g_vals[i] else 1.0 for i in range(len(g_vals))]

N      = len(radar_metrics)
angles = [n/float(N)*2*np.pi for n in range(N)] + [0]
g_norm_c = g_norm + g_norm[:1]
h_norm_c = h_norm + h_norm[:1]

fig, ax = plt.subplots(1,1, figsize=(7,7), subplot_kw=dict(polar=True))
ax.plot(angles, g_norm_c, "o-", linewidth=2, color=GPU_COLOR,    label="GPU-only (baseline=1.0)")
ax.fill(angles, g_norm_c, alpha=0.12, color=GPU_COLOR)
ax.plot(angles, h_norm_c, "s--", linewidth=2, color=HYBRID_COLOR, label="Hybrid")
ax.fill(angles, h_norm_c, alpha=0.12, color=HYBRID_COLOR)
ax.set_xticks(angles[:-1]); ax.set_xticklabels(radar_metrics, fontsize=11)
ax.set_yticks([0.25,0.5,0.75,1.0,1.25])
ax.set_yticklabels(["0.25","0.50","0.75","1.00","1.25"], fontsize=8)
ax.set_title(f"Overall Performance Radar\n{latest['name']}  [{model_name}]",
             fontsize=12, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3,1.1), framealpha=0.85)
ax.grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

print("\nRadar values (GPU-only = 1.0 baseline):")
for m, hv in zip(radar_metrics, h_norm):
    mark = "✓" if hv>=1.0 else "✗"
    print(f"  {mark} {m:15s}: GPU-only=1.000  Hybrid={hv:.3f}")


In [ ]:
# ============================================================
#  Section 12 ▶ GPU-only Solo Runs (no comparison)
# ============================================================

solo_runs = [r for r in runs if "comparison" not in r]
if not solo_runs:
    print("No solo runs found.")
else:
    solo_rows = []
    for r in solo_runs:
        for mode in ("gpu_only","hybrid"):
            d = r.get(mode)
            if d is None: continue
            solo_rows.append({
                "Run":r["name"], "Mode":mode,
                "Model":d.get("model_id","?").split("/")[-1],
                "Prompts":d.get("num_prompts"),
                "Completed":d.get("completed"),
                "Req TP":round(d.get("request_throughput",0),2),
                "Out TP":round(d.get("output_throughput",0),1),
                "Mean TTFT (ms)":round(d.get("mean_ttft_ms",0),1),
                "Mean TPOT (ms)":round(d.get("mean_tpot_ms",0),2),
            })
    solo_df = pd.DataFrame(solo_rows)
    display(solo_df.style
        .format({"Req TP":"{:.2f}","Out TP":"{:.0f}",
                 "Mean TTFT (ms)":"{:.0f}","Mean TPOT (ms)":"{:.2f}"})
        .set_table_styles([{"selector":"th","props":[("background-color","#2d4a7a"),
                                                      ("color","white"),("padding","5px 10px")]}])
        .set_caption("<b>GPU-only Solo Runs</b>"))

    gpu_solo = solo_df[solo_df["Mode"]=="gpu_only"]
    if not gpu_solo.empty:
        fig, ax = plt.subplots(figsize=(10,4))
        sns.barplot(data=gpu_solo, x="Run", y="Out TP", hue="Model",
                    palette="muted", ax=ax, edgecolor="white", linewidth=0.8, width=0.55)
        ax.set_title("GPU-only Solo Runs — Output Throughput", fontweight="semibold")
        ax.set_ylabel("Output Throughput (tok/s)"); ax.set_xlabel("")
        ax.tick_params(axis="x", rotation=15)
        for patch in ax.patches:
            v = patch.get_height()
            if v>0: ax.text(patch.get_x()+patch.get_width()/2, v+30, f"{v:.0f}",
                            ha="center", fontsize=9)
        sns.despine(); plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
#  Section 13 ▶ Key Insights Summary
# ============================================================

print("=" * 60)
print("  vLLM Hybrid Benchmark — Key Insights")
print("=" * 60)

if cmp_runs:
    all_speedups = [r["comparison"].get("comparison",{}).get("output_tok_speedup", np.nan)
                    for r in cmp_runs]
    valid_sp  = [s for s in all_speedups if not np.isnan(s)]
    avg_sp    = np.mean(valid_sp)  if valid_sp else np.nan
    best_sp   = max(valid_sp)      if valid_sp else np.nan
    best_run  = cmp_runs[valid_sp.index(best_sp)] if valid_sp else None

    print(f"\n[Throughput]")
    print(f"  Avg Output Token Speedup : {avg_sp:.3f}x  (vs GPU-only)")
    if best_run:
        bm = best_run.get("hybrid",best_run.get("gpu_only",{})).get("model_id","?").split("/")[-1]
        print(f"  Best Speedup             : {best_sp:.3f}x  [{best_run['name']}, {bm}]")

    ttft_diffs = [r["comparison"].get("comparison",{}).get("mean_ttft_ms",{}).get("diff_pct",np.nan)
                  for r in cmp_runs]
    valid_ttft = [v for v in ttft_diffs if not np.isnan(v)]
    avg_ttft   = np.mean(valid_ttft) if valid_ttft else np.nan
    print(f"\n[Latency]")
    print(f"  Avg TTFT change (Hybrid vs GPU) : {avg_ttft:+.1f}%  (positive = Hybrid is slower)")

    cpu_g_avgs, cpu_h_avgs = [], []
    for r in cmp_runs:
        cu = r.get("comparison",{}).get("cpu_utilization",{})
        cpu_g_avgs.append(cu.get("gpu_only",{}).get("cpu_avg_util_pct",{}).get("mean",np.nan))
        cpu_h_avgs.append(cu.get("hybrid",  {}).get("cpu_avg_util_pct",{}).get("mean",np.nan))
    vg = [v for v in cpu_g_avgs if not np.isnan(v)]
    vh = [v for v in cpu_h_avgs if not np.isnan(v)]
    print(f"\n[CPU Utilization]")
    if vg: print(f"  GPU-only mode avg CPU util : {np.mean(vg):.1f}%")
    if vh: print(f"  Hybrid   mode avg CPU util : {np.mean(vh):.1f}%")

    print(f"\n[Statistics]")
    print(f"  Total runs             : {len(runs)}")
    print(f"  Runs with comparison   : {len(cmp_runs)}")
    print(f"  Models tested          : {list(model_groups.keys())}")
    print(f"  Date range             : {runs[0]['ts'].strftime('%Y-%m-%d')} ~ {runs[-1]['ts'].strftime('%Y-%m-%d')}")

print("\n" + "=" * 60)
